# Object Localization Practice

**Topic:** Advanced Computer Vision &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `X_train` `(3000, 20, 20)`, `Y_train` `(3000, 2)`: images and their true blob centers as `(row, col)` floats.
- `X_test` `(800, 20, 20)`: predict centers for these. The true centers are used by the evaluation cell.
- Distractors: a dimmer second blob in some images and streak noise. The blob's peak is broad and the background is noisy, so a plain argmax can land a pixel or two off the true center.

## Your task

1. Build a localizer: smooth-then-argmax with subpixel refinement, an intensity-weighted centroid around the peak, or a PyTorch CNN with an MSE head trained on `Y_train`.
2. Set `predictions` to a `(800, 2)` array of predicted `(row, col)` centers.

## Evaluation

Mean Euclidean error over the test set must be below 1.5 pixels.


In [ ]:
import numpy as np

def blobs(n, seed):
    r = np.random.default_rng(seed)
    X = r.normal(0.1, 0.05, (n, 20, 20)).astype(np.float32).clip(0, 1)
    Y = np.zeros((n, 2), dtype=np.float32)
    rows, cols = np.mgrid[0:20, 0:20]
    for i in range(n):
        cy, cx = r.uniform(4, 16, 2)
        X[i] += 0.9 * np.exp(-(((rows - cy) ** 2 + (cols - cx) ** 2) / (2 * 1.6 ** 2)))
        if r.random() < 0.4:                       # dim distractor blob
            dy, dx = r.uniform(2, 18, 2)
            X[i] += 0.35 * np.exp(-(((rows - dy) ** 2 + (cols - dx) ** 2) / (2 * 1.3 ** 2)))
        if r.random() < 0.3:                       # streak noise
            X[i, r.integers(0, 20), :] += r.uniform(0.1, 0.25)
        Y[i] = (cy, cx)
    return X.clip(0, 1.6), Y

X_train, Y_train = blobs(3000, 411)
X_test = blobs(800, 412)[0]
print(f"X_train {X_train.shape}, centers Y_train {Y_train.shape}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: mean localization error in pixels
# =====================================================================
import numpy as np

_Y_test_true = blobs(800, 412)[1]

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions, dtype=float)
if _pred.shape != _Y_test_true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_Y_test_true.shape}.")

_err = np.linalg.norm(_pred - _Y_test_true, axis=1)
print("=" * 52)
print(f"Mean error   : {float(_err.mean()):.3f} px   (target: < 1.5)")
print(f"Median error : {float(np.median(_err)):.3f} px")
print(f"Within 2 px  : {float((_err < 2).mean()) * 100:.1f}% of images")
print("=" * 52)
print("PASSED: sharp eyes!" if float(_err.mean()) < 1.5 else "NOT PASSED yet: the peak is broad and the background is noisy, so refine around the brightest pixel instead of taking it as the center.")


## Solution

Smooth each image a little (to suppress streaks), take the brightest pixel as a coarse peak, then refine with an intensity-weighted centroid in a small window around it. The true blob is brighter than the distractor, so the smoothed argmax picks the right one.

```python
import numpy as np

def box_blur(img, k=3):
    pad = k // 2
    padded = np.pad(img, pad, mode="edge")
    out = np.zeros_like(img)
    for dy in range(k):
        for dx in range(k):
            out += padded[dy:dy + 20, dx:dx + 20]
    return out / (k * k)

rows, cols = np.mgrid[0:20, 0:20]
preds = np.zeros((len(X_test), 2), dtype=float)
for i, img in enumerate(X_test):
    sm = box_blur(img.astype(float))
    peak = np.unravel_index(int(sm.argmax()), sm.shape)
    win = ((np.abs(rows - peak[0]) <= 3) & (np.abs(cols - peak[1]) <= 3))
    wts = np.where(win, sm, 0.0)
    wts = np.maximum(wts - wts[win].min(), 0.0)     # remove background level
    preds[i, 0] = (rows * wts).sum() / wts.sum()
    preds[i, 1] = (cols * wts).sum() / wts.sum()

predictions = preds
```
